<a href="https://colab.research.google.com/github/carlos-osorio/monitor-fletes/blob/main/notebooks/00_ingesta_mensual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ingesta mensual — monitor-fletes

Correr una vez al mes, cuando el RNDC publique el archivo del mes anterior
(rezago típico: ~2 meses).

## Procedimiento
1. Ir a rndc.mintransporte.gov.co → Consultas Públicas → responder la suma.
2. En "Año Mes" escribir AAAAMM y descargar "estadísticas de rutas y
   mercancías → del Mes".
3. Guardar el archivo **sin renombrar** en Drive:
   `MyDrive/monitor-fletes/datos/`. Si el navegador le pone " (1)",
   corregir el nombre — el script lo rechaza a propósito.
4. Correr las celdas de abajo en orden.
5. Subir el Parquet nuevo al repo (última celda).



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone https://github.com/carlos-osorio/monitor-fletes.git

Cloning into 'monitor-fletes'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 68 (delta 26), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 64.54 KiB | 1.40 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [ ]:
!python monitor-fletes/src/ingestar.py \
  "/content/drive/MyDrive/monitor-fletes/datos" \
  "/content/drive/MyDrive/monitor-fletes/procesado"

  OMITE: EstadisticasRNDC_202412 (1).xlsx no calza el patrón EstadisticasRNDC_AAAAMM.xlsx

Listo: 0 mes(es) nuevo(s) procesado(s).


In [2]:
from google.colab import userdata
import shutil, subprocess, glob, os
from pathlib import Path

TOKEN = userdata.get('GH_TOKEN')
REPO  = "carlos-osorio/monitor-fletes"
ORIGEN = "/content/drive/MyDrive/monitor-fletes/procesado"

# Clon limpio con credenciales embebidas en la URL (no queda en el notebook)
!rm -rf /content/repo
subprocess.run(["git", "clone", f"https://{TOKEN}@github.com/{REPO}.git",
                "/content/repo"], check=True, capture_output=True)

destino = Path("/content/repo/data/procesado")
destino.mkdir(parents=True, exist_ok=True)

nuevos = []
for p in sorted(glob.glob(f"{ORIGEN}/fletes_*.parquet")):
    d = destino / os.path.basename(p)
    if not d.exists():
        shutil.copy(p, d)
        nuevos.append(d.name)

if not nuevos:
    print("Sin extractos nuevos que publicar.")
else:
    print("Nuevos:", ", ".join(nuevos))
    %cd /content/repo
    !git config user.name "ingesta-bot"
    !git config user.email "actions@github.com"
    !git add data/procesado/
    !git commit -m "Ingesta: {', '.join(nuevos)}"
    !git pull --rebase origin main
    !git push
    %cd /content

Sin extractos nuevos que publicar.
